# Generate the GPT-2 Joke Arena bank

This performs **real inference** and creates exactly:

- **10 prompts**
- **8 answers per prompt**
- **2 GPT-2 Small**
- **2 GPT-2 Medium**
- **2 GPT-2 Large**
- **2 GPT-2 XL**

Total: **80 generated continuations**.

Each model is loaded once, generates its 20 answers, then is unloaded before the next model. EOS is natural; nothing is forced to a fixed word count.

At the end, download `joke_bank.json` and replace the starter file with the same name in the GitHub Pages repo.

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate safetensors

In [ ]:
import gc, hashlib, json, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Settings

In [ ]:
PROMPTS = ['I went to the store yesterday, but', 'When I opened the refrigerator this morning,', 'The new employee seemed completely normal until', 'Nobody at the restaurant could explain why', 'My neighbor knocked on my door and asked if', 'The package arrived three weeks late, and', 'I knew the hotel was unusual when', 'At first the job interview was going well, but', 'The sign on the door clearly said', 'I called customer service because']

In [ ]:
MODELS = [
    ("small",  "GPT-2 Small",  "openai-community/gpt2"),
    ("medium", "GPT-2 Medium", "openai-community/gpt2-medium"),
    ("large",  "GPT-2 Large",  "openai-community/gpt2-large"),
    ("xl",     "GPT-2 XL",     "openai-community/gpt2-xl"),
]

MAX_NEW_TOKENS = 220
TEMPERATURE = 0.95
TOP_P = 0.95
TOP_K = 50

MASTER_SEED = 20260819
CHECKPOINT = Path("joke_bank_checkpoint.json")
OUTPUT = Path("joke_bank.json")

len(PROMPTS), MODELS

## Deterministic generation plan

Each prompt/model gets two fixed seeds. Re-running with the same settings produces the same planned trials.

In [ ]:
def stable_seed(prompt_index, difficulty, replicate):
    s = f"{MASTER_SEED}|{prompt_index}|{difficulty}|{replicate}"
    return int(hashlib.sha256(s.encode()).hexdigest()[:8], 16)

plan = []
for pi, prompt in enumerate(PROMPTS, start=1):
    for difficulty, label, model_id in MODELS:
        for rep in [1, 2]:
            plan.append({
                "prompt_id": f"p{pi:02d}",
                "prompt_index": pi,
                "prompt": prompt,
                "difficulty": difficulty,
                "model_label": label,
                "model_id": model_id,
                "replicate": rep,
                "seed": stable_seed(pi, difficulty, rep),
                "answer_id": f"p{pi:02d}_{difficulty}_{rep}",
            })

plan_df = pd.DataFrame(plan)
display(plan_df.head(16))
print("Total planned answers:", len(plan_df))

## Generation helpers

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_model(model_id):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    mdl.eval()
    return tok, mdl

@torch.inference_mode()
def generate_one(tokenizer, model, prompt, seed):
    seed_everything(seed)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    t0 = time.perf_counter()
    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    dt = time.perf_counter() - t0

    new_ids = out[0, input_ids.shape[1]:]
    continuation = tokenizer.decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    natural_eos = (
        tokenizer.eos_token_id is not None
        and tokenizer.eos_token_id in new_ids.tolist()
    )
    return {
        "text": prompt + continuation,
        "continuation": continuation,
        "generated_tokens": int(len(new_ids)),
        "generation_seconds": dt,
        "natural_eos": natural_eos,
    }

## Generate all 80 answers

The checkpoint is rewritten after every successful answer. If the runtime stops but you still have the checkpoint file, rerunning the cell skips completed answer IDs.

For a durable resume across Colab sessions, save the checkpoint to Google Drive or download it periodically.

In [ ]:
if CHECKPOINT.exists():
    generated = json.loads(CHECKPOINT.read_text(encoding="utf-8"))
else:
    generated = {}

for difficulty, model_label, model_id in MODELS:
    todo = [
        x for x in plan
        if x["difficulty"] == difficulty and x["answer_id"] not in generated
    ]
    if not todo:
        print(model_label, "already complete")
        continue

    print("\n" + "="*90)
    print(model_label, model_id, "-", len(todo), "remaining")
    print("="*90)

    load_start = time.perf_counter()
    tokenizer, model = load_model(model_id)
    print(f"Loaded in {time.perf_counter()-load_start:.1f}s")

    for n, item in enumerate(todo, start=1):
        result = generate_one(tokenizer, model, item["prompt"], item["seed"])
        generated[item["answer_id"]] = {
            **item,
            **result,
        }
        CHECKPOINT.write_text(
            json.dumps(generated, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(
            f"[{n:02d}/{len(todo):02d}] {item['answer_id']}  "
            f"{result['generated_tokens']} tok  "
            f"{result['generation_seconds']:.2f}s  "
            f"EOS={result['natural_eos']}"
        )

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nGenerated:", len(generated), "/", len(plan))

## Validate and build `joke_bank.json`

In [ ]:
assert len(generated) == 80, f"Expected 80 answers, found {len(generated)}"

bank = {
    "schema_version": 1,
    "generated": True,
    "generation_settings": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "master_seed": MASTER_SEED,
    },
    "models": [
        {"difficulty": d, "label": label, "model_id": mid}
        for d, label, mid in MODELS
    ],
    "prompts": [],
}

for pi, prompt in enumerate(PROMPTS, start=1):
    pid = f"p{pi:02d}"
    answers = []
    for difficulty, label, model_id in MODELS:
        for rep in [1, 2]:
            aid = f"{pid}_{difficulty}_{rep}"
            g = generated[aid]
            answers.append({
                "answer_id": aid,
                "difficulty": difficulty,
                "model_label": label,
                "model_id": model_id,
                "replicate": rep,
                "seed": g["seed"],
                "text": g["text"],
                "continuation": g["continuation"],
                "generated_tokens": g["generated_tokens"],
                "generation_seconds": g["generation_seconds"],
                "natural_eos": g["natural_eos"],
            })
    assert len(answers) == 8
    bank["prompts"].append({
        "prompt_id": pid,
        "text": prompt,
        "answers": answers,
    })

assert len(bank["prompts"]) == 10
for p in bank["prompts"]:
    counts = {}
    for a in p["answers"]:
        counts[a["difficulty"]] = counts.get(a["difficulty"], 0) + 1
    assert counts == {"small":2, "medium":2, "large":2, "xl":2}, counts

OUTPUT.write_text(json.dumps(bank, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", OUTPUT, "-", OUTPUT.stat().st_size/1024, "KB")

## Timing summary

In [ ]:
rows = list(generated.values())
df = pd.DataFrame(rows)

summary = df.groupby("model_label").agg(
    answers=("answer_id","count"),
    median_seconds=("generation_seconds","median"),
    mean_seconds=("generation_seconds","mean"),
    median_tokens=("generated_tokens","median"),
    natural_eos_rate=("natural_eos","mean"),
).reset_index()

display(summary)
print("Total generation seconds:", round(df["generation_seconds"].sum(), 1))
print("Total generation minutes:", round(df["generation_seconds"].sum()/60, 2))

## Download the finished bank

In [ ]:
from google.colab import files
files.download("joke_bank.json")